# TinyLlama Roleplay Training Pipeline (Multi-Stage, Refactored)

Breakdown:
1. Environment setup
2. Config
3. Training (Stage1 SFT, Stage2 SFT, optional Stage3 DPO)
4. Inference stack smoke test (memory + prompt + world + emotion)


## 1) Environment setup

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes sentencepiece faiss-cpu sentence-transformers PyYAML


In [ ]:
import sys
sys.path.append('.')


## 2) Config

In [ ]:
MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_ROOT = '/content/tinyllama-roleplay-multistage'
MAX_LENGTH = 1024
SEED = 42


## 3) Training (multi-stage)

Preserves the original notebook behavior:
- Stage 1 SFT: `OpenAssistant/oasst1`, `RyokoAI/ShareGPT52K`
- Stage 2 SFT: `AlekseyKorshuk/persona-chat`, `Anthropic/hh-rlhf`
- Stage 3 DPO: `Anthropic/hh-rlhf` (if available)
- Exports: adapter, tokenizer, merged model


In [ ]:
from training.multistage_pipeline import run_multistage_pipeline
from utils.logging import configure_logging

logger = configure_logging('INFO', 'roleplay_multistage')
run_multistage_pipeline(
    model_id=MODEL_ID,
    output_root=OUTPUT_ROOT,
    max_length=MAX_LENGTH,
    seed=SEED,
    trust_remote_code=True,
    logger=logger,
)


## 4) Inference stack smoke test

This demonstrates long-term memory retrieval and prompt construction.
For actual generation, attach a backend (`TransformersBackend`, `LlamaCppBackend`, etc.).


In [ ]:
from characters.profile import load_character_profile_by_id
from emotion_engine.engine import EmotionState
from memory.faiss_store import FaissMemoryStore
from prompt_builder.builder import PromptBuilder
from world_state.state import WorldState

character_id = 'villain'
character = load_character_profile_by_id(character_id)
world = WorldState(location='Capital City', time='midnight', characters_present=[character.name], current_event='A masked gala', story_progress='in_progress')
emotions = EmotionState(by_character={character_id: 'suspicious'})

memory = FaissMemoryStore(persist_dir=None, max_records=1000)
memory.add('user123', 'User promised to bring the obsidian key to the gala.', meta={'type': 'story_event'})
memory.add('user123', 'Lady Vesper hinted the Duke is compromised.', meta={'type': 'story_event'})

retrieved = [r.text for r in memory.retrieve('obsidian key', user_id='user123', k=4)]
pb = PromptBuilder()
prompt = pb.build(
    character=character,
    world_state=world,
    emotion_state=emotions,
    character_id=character_id,
    retrieved_memories=retrieved,
    chat_history=[('I have the key.', 'Then you understand the price of secrets.')],
    user_input='What do you want from me tonight?'
)
print(prompt[:2000])
